# Meta-labeling

Meta-labeling (Lopez de Prado 2018) splits a trading decision in two. A rule
gives the side of each trade, long or short. A second model, trained on the
outcome of the rule's past trades, gives the probability that a trade works,
and only the trades above a threshold are taken. The outcome of a trade is a
[`TripleBarrier`](../functions_fin/TripleBarrier.md) label: with symmetric
barriers, the trade worked when the label equals its side. The
[labels guide](../labels.md) shows the targets for other kinds of prediction.

The market is synthetic. A regime variable `g` changes slowly. When `g` is
positive, the price moves back toward its moving average; when `g` is negative,
it moves away from it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from screamer import (TripleBarrier, EwMean, EwStd, LogReturn, Lag, BacktestPriceTarget,
                      backtest_report, forecast_pairs)

# g is an AR(1) regime variable with unit variance. f and s are fast and slow
# exponential moving averages of the log price x. The log return at step t has a
# drift of 0.3 * tanh(2 * g[t - 1]) * 0.01 toward s when g > 0, away from s when
# g < 0, a weak pull toward x = 0, and noise of 0.01.
rng = np.random.default_rng(25)
n = 30000
g = np.zeros(n)
for t in range(1, n):
    g[t] = 0.995 * g[t - 1] + np.sqrt(1 - 0.995 ** 2) * rng.standard_normal()
x = np.zeros(n)
f = s = 0.0
for t in range(1, n):
    f += 0.5 * (x[t - 1] - f)
    s += 2 / 21 * (x[t - 1] - s)
    x[t] = (x[t - 1] - 0.3 * np.tanh(2 * g[t - 1]) * np.sign(f - s) * 0.01
            - 0.001 * x[t - 1] + 0.01 * rng.standard_normal())
close = 100 * np.exp(x)
step = np.arange(n)

## The rule

The rule compares a fast and a slow exponential moving average of the price,
with spans 3 and 20. Its side is short (`-1`) while the fast average is above
the slow one and long (`+1`) while it is below. An entry is a step at which the
side changes.

In [ ]:
fast = EwMean(span=3)(close)
slow = EwMean(span=20)(close)
side = -np.sign(fast - slow)
entry = np.r_[False, side[1:] != side[:-1]] & (side != 0)
print(f"{entry.sum()} entries in {n} steps")

## Meta-labels

`TripleBarrier` labels each step with barriers half an expected deviation
above and below the entry price, over `h = 10` steps. A trade worked when the
label equals its side: a long whose price reaches the upper barrier first, or
a short whose price reaches the lower barrier first. A label of `0` counts as
not worked.

The output at step `t` is the entry at `t - h`, so it is compared with the side
at `t - h`, `Lag(h)(side)`. `meta` is `1` when the trade worked, `0` when it did
not, and `NaN` while the label is unknown.

In [ ]:
h = 10
out = TripleBarrier(horizon=h, upper=0.5, lower=0.5, barrier_unit="vol")(close)
worked = out[:, 0] == Lag(h)(side)
meta = np.where(np.isnan(out[:, 0]), np.nan, worked.astype(float))

## Features

`X` has four columns: the regime variable `g`, the distance between the two
averages in units of the one-step volatility, that volatility, and the return
over the last `h` steps. `forecast_pairs(X, meta, count=h)` pairs row `t` with
the meta-label of the entry at `t`. Without `dropna`, row `t` is step `t`, and
`rows` keeps the entries whose features and meta-label are known.

In [ ]:
vol = EwStd(span=100)(LogReturn(1)(close))
X = np.column_stack([
    g,
    (fast - slow) / slow / vol,          # distance between the averages
    vol,                                 # one-step volatility
    LogReturn(h)(close),                 # return over the last h steps
])
Xp, yp = forecast_pairs(X, meta, count=h)
rows = entry & ~np.isnan(Xp).any(axis=1) & ~np.isnan(yp)
print(f"{rows.sum()} trades, {np.mean(yp[rows]):.1%} worked")

## Train and test split

The first 60% of the steps are for training and the rest for testing. The
training rows stop `h` steps before the test block, so no training label uses a
price from the test block. The model is a logistic regression on standardized
features.

In [ ]:
cut = int(0.6 * n)
train = rows & (step < cut - h)
test = rows & (step >= cut)

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(Xp[train], yp[train])
print("coefficients (g, distance, volatility, last return)")
print(model[-1].coef_.round(3))

The coefficient of `g` is positive: the higher `g`, the higher the predicted
probability that a trade works.

## Which trades to take

`predict_proba` gives each test trade's probability of working, from the
features at its entry. A trade is taken when that probability is above 0.5.

In [ ]:
p = model.predict_proba(Xp[test])[:, 1]
take = p > 0.5
print(f"taken {take.sum()} of {test.sum()} test trades")
print(f"worked: {np.mean(yp[test][take]):.1%} of the taken trades, "
      f"{np.mean(yp[test][~take]):.1%} of the skipped ones")

## Backtest with and without the filter

A trade holds one unit on its side from the entry until its exit, at the first
barrier or after `h` steps. The exit needs only the prices up to it, and
`holding` (column 2 of `out`, stamped `h` steps after the entry) gives its step.
A new trade replaces an open one. `BacktestPriceTarget` charges a fee of 5
basis points on the traded notional, and a position set at step `t` earns the
price change from `t` to `t + 1`.

In [ ]:
def positions(entries):
    pos = np.zeros(n)
    for e in entries:
        pos[e:e + int(out[e + h, 2])] = side[e]
    return pos

test_entries = step[test]
results = {}
for name, entries in [("every trade", test_entries), ("filtered", test_entries[take])]:
    running, summary = backtest_report(BacktestPriceTarget(fee=5e-4)(positions(entries)[cut:], close[cut:]))
    results[name] = running
    print(f"{name:12s} " + "  ".join(f"{key} {summary[key]:.4g}" for key in
                                     ["total_pnl", "total_cost", "num_trades", "max_drawdown"]))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for (name, running), color in zip(results.items(), ["#999999", "#0074a2"]):
    ax.plot(step[cut:], running["equity"], color=color, label=name)
ax.axhline(0, color="k", lw=0.6)
ax.set(title="equity on the test block", xlabel="step", ylabel="cumulative pnl")
ax.legend()
fig.tight_layout()

## Asymmetric barriers

With a profit target `u` and a stop `d` that differ, the barriers depend on the
side: a long takes profit at the upper barrier and a short at the lower one.
Two `TripleBarrier` operators label both cases, and each row takes the label of
its entry's side. The trade worked when that label equals the side.

```python
u, d = 1.0, 0.5
long_label = TripleBarrier(horizon=h, upper=u, lower=d, barrier_unit="vol")(close)[:, 0]
short_label = TripleBarrier(horizon=h, upper=d, lower=u, barrier_unit="vol")(close)[:, 0]
entry_side = Lag(h)(side)
label = np.where(entry_side > 0, long_label, short_label)
meta = np.where(np.isnan(label), np.nan, (label == entry_side).astype(float))
```